# Parallel BioEmu Colab Worker

This notebook runs one immutable shard. Each BioEmu bundle is uploaded to a Hugging Face dataset repository and remotely verified. The completion marker is uploaded last. After every task in the shard has a verified marker, the managed Colab runtime is disconnected and deleted automatically.

Failures intentionally leave the runtime attached so logs and incomplete local files remain available for recovery. If Colab later reclaims the VM, an external actor must obtain a new runtime and rerun this notebook or `supervise_worker.py`; the notebook cannot restart a dead VM or automate Google account login. Add `HF_TOKEN` as a Colab secret; never paste it into a notebook cell.

## 1. Install

Clone or upload this repository to `/content/protein-conformation-guided-random-walk`, then run this cell.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/protein-conformation-guided-random-walk')
if not REPO_ROOT.is_dir():
    raise FileNotFoundError(f'Upload or clone the repository at {REPO_ROOT}')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_ROOT}/parallel_bioemu_tools[huggingface]', 'bioemu[cuda]'],
    check=True,
)
print('Worker environment installed.')

## 2. Configure

Upload one generated shard manifest to `/content/shard_000.json`. Set the private Hugging Face dataset repository below. `AUTO_RELEASE_RUNTIME=True` is the credit-saving default. Keep one task per shard for `T4` runs because BioEmu still lacks mid-sample checkpointing.

In [ ]:
import os
from google.colab import userdata

from parallel_bioemu_tools.colab_gpu import ensure_allowed_gpu

SHARD_MANIFEST = Path('/content/shard_000.json')
WORKSPACE_ROOT = Path('/content/parallel_bioemu_worker')
HF_REPO_ID = 'your-account/your-private-dataset'
AUTO_RELEASE_RUNTIME = True
ALLOWED_GPU_TYPES = ('L4', 'T4')

if not SHARD_MANIFEST.is_file():
    raise FileNotFoundError(f'Upload the shard manifest to {SHARD_MANIFEST}')
gpu_report = ensure_allowed_gpu(ALLOWED_GPU_TYPES)
print(gpu_report.describe())
token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets and enable notebook access.')
os.environ['HF_TOKEN'] = token
print('Configuration validated; token loaded without displaying it.')

## 3. Run The Shard

Do not put runtime release in a `finally` block. This cell releases only after `run_worker_shard` confirms every remote completion marker. If the runtime dies before the final marker upload, rerun the same shard on a newly assigned runtime. If the marker is already durable in Hugging Face, the rerun will detect it and skip the completed task.

In [ ]:
from parallel_bioemu_tools.colab_runtime import release_colab_runtime
from parallel_bioemu_tools.hf_storage import HuggingFaceStorage
from parallel_bioemu_tools.supervisor import SupervisorConfig, run_supervised_shard

storage = HuggingFaceStorage(repo_id=HF_REPO_ID)
try:
    result = run_supervised_shard(
        SHARD_MANIFEST,
        storage=storage,
        config=SupervisorConfig(
            workspace_root=WORKSPACE_ROOT,
            max_attempts=3,
            retry_delay_seconds=30.0,
            allowed_gpu_types=ALLOWED_GPU_TYPES,
        ),
    )
except Exception:
    print('Shard failed; runtime remains attached for inspection or retry.')
    raise

print(result.gpu_report.describe())
print(
    f'Remotely verified {len(result.completions)} task(s) after {result.attempts_used} attempt(s).'
)
if AUTO_RELEASE_RUNTIME:
    print('Releasing managed Colab runtime now.', flush=True)
    release_colab_runtime()
else:
    print('AUTO_RELEASE_RUNTIME is disabled; release the runtime manually.')